# InstantID Generation Server

Runs an identity-conditioned image generation API on Kaggle T4 GPU and exposes it
via a cloudflared tunnel for use with `scripts/instantid_client.py`.

**Before running:**
- Accelerator: Settings → Accelerator → GPU T4 x1
- Internet: Settings → Internet → On

**After running all cells**, copy the printed `INSTANTID_SERVER_URL` into your `.env` file.
The tunnel stays alive as long as this notebook session is active (~9hr on Kaggle free tier).

In [ ]:
# Cell 1 — Install dependencies
import subprocess, sys

packages = [
    'insightface',
    'onnxruntime-gpu',
    'diffusers>=0.25.0',
    'accelerate',
    'transformers',
    'controlnet-aux',
    'flask',
    'huggingface_hub',
    'opencv-python-headless',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + packages, check=True)
print('Dependencies installed.')

In [ ]:
# Cell 2 — Download InstantID weights and custom pipeline
import os, urllib.request
from huggingface_hub import hf_hub_download, snapshot_download

os.makedirs('./checkpoints', exist_ok=True)

# Base URL for the InstantID repo (moved from InstantX → instantX-research org)
GH_RAW = 'https://raw.githubusercontent.com/instantX-research/InstantID/main'

# Download the custom pipeline script
urllib.request.urlretrieve(
    f'{GH_RAW}/pipeline_stable_diffusion_xl_instantid.py',
    'pipeline_stable_diffusion_xl_instantid.py',
)
print('Custom pipeline script downloaded.')

# Download the ip_adapter module (imported by the pipeline script)
os.makedirs('./ip_adapter', exist_ok=True)
for fname in ['__init__.py', 'attention_processor.py', 'resampler.py', 'utils.py']:
    try:
        urllib.request.urlretrieve(f'{GH_RAW}/ip_adapter/{fname}', f'./ip_adapter/{fname}')
        print(f'  ip_adapter/{fname} downloaded.')
    except Exception as e:
        # __init__.py may not exist; that's fine
        open(f'./ip_adapter/{fname}', 'w').close()
        print(f'  ip_adapter/{fname} created empty ({e})')

# InstantID IP-Adapter weights
hf_hub_download(
    repo_id='InstantX/InstantID',
    filename='ip-adapter.bin',
    local_dir='./checkpoints',
)
print('ip-adapter.bin downloaded.')

# InstantID ControlNet weights (~2.5 GB)
snapshot_download(
    repo_id='InstantX/InstantID',
    allow_patterns=['ControlNetModel/*'],
    local_dir='./checkpoints',
)
print('ControlNet weights downloaded.')

In [ ]:
# Cell 3 — (skipped) Base model loads directly from HuggingFace hub in Cell 4
# Downloading 6.9 GB to /kaggle/working fills the 20 GB quota before model loading.
# diffusers from_pretrained caches to ~/.cache/huggingface which has a separate allocation.
print('Skipping pre-download — Cell 4 will load from hub with fp16.')

In [ ]:
# Cell 4 — Load InsightFace + InstantID pipeline
import sys
sys.path.insert(0, '.')

import torch
import cv2
import numpy as np
from PIL import Image
from insightface.app import FaceAnalysis
from diffusers.models import ControlNetModel
from diffusers import AutoencoderKL
from pipeline_stable_diffusion_xl_instantid import (
    StableDiffusionXLInstantIDPipeline,
    draw_kps,
)

print('Loading InsightFace (buffalo_l)...')
face_app = FaceAnalysis(
    name='buffalo_l',
    root='/tmp/insightface',
    providers=['CUDAExecutionProvider', 'CPUExecutionProvider'],
)
face_app.prepare(ctx_id=0, det_size=(640, 640))
print('InsightFace loaded.')

print('Loading ControlNet...')
controlnet = ControlNetModel.from_pretrained(
    './checkpoints/ControlNetModel',
    torch_dtype=torch.float16,
)

# fp16-fixed VAE prevents NaN artifacts at half precision
print('Loading VAE...')
vae = AutoencoderKL.from_pretrained(
    'madebyollin/sdxl-vae-fp16-fix',
    torch_dtype=torch.float16,
)

# Load base model directly from hub with fp16 — caches to ~/.cache (separate quota)
print('Loading pipeline from hub (fp16)...')
pipe = StableDiffusionXLInstantIDPipeline.from_pretrained(
    'stabilityai/stable-diffusion-xl-base-1.0',
    controlnet=controlnet,
    vae=vae,
    torch_dtype=torch.float16,
    variant='fp16',
    use_safetensors=True,
    safety_checker=None,
)
pipe.cuda()
pipe.load_ip_adapter_instantid('./checkpoints/ip-adapter.bin')
print('Pipeline ready.')

In [ ]:
# Cell 5 — Generation function + Flask API
import base64, io, threading
from flask import Flask, request, jsonify


def generate_images(
    face_image: Image.Image,
    positive_prompt: str,
    negative_prompt: str = '',
    num_images: int = 1,
    guidance_scale: float = 7.0,
    seed: int | None = None,
    ip_adapter_scale: float = 0.8,
    controlnet_scale: float = 0.8,
) -> list[Image.Image]:
    face_np = cv2.cvtColor(np.array(face_image), cv2.COLOR_RGB2BGR)
    faces = face_app.get(face_np)
    if not faces:
        raise ValueError('No face detected in reference image')
    # Use largest detected face
    face = sorted(faces, key=lambda f: (
        (f['bbox'][2] - f['bbox'][0]) * (f['bbox'][3] - f['bbox'][1])
    ))[-1]
    face_emb = face['embedding']
    face_kps = draw_kps(face_image, face['kps'])

    pipe.set_ip_adapter_scale(ip_adapter_scale)
    generator = (
        torch.Generator('cuda').manual_seed(seed)
        if seed is not None else None
    )
    result = pipe(
        prompt=positive_prompt,
        negative_prompt=negative_prompt,
        image_embeds=face_emb,
        image=face_kps,
        controlnet_conditioning_scale=controlnet_scale,
        num_inference_steps=30,
        guidance_scale=guidance_scale,
        num_images_per_prompt=num_images,
        generator=generator,
    )
    return result.images


flask_app = Flask(__name__)

@flask_app.route('/health')
def health():
    return jsonify({'status': 'ok', 'gpu': torch.cuda.is_available()})

@flask_app.route('/generate', methods=['POST'])
def generate_endpoint():
    data = request.json
    try:
        face_bytes = base64.b64decode(data['face_image_b64'])
        face_image = Image.open(io.BytesIO(face_bytes)).convert('RGB')
        images = generate_images(
            face_image=face_image,
            positive_prompt=data['positive_prompt'],
            negative_prompt=data.get('negative_prompt', ''),
            num_images=data.get('num_images', 1),
            guidance_scale=data.get('guidance_scale', 7.0),
            seed=data.get('seed'),
            ip_adapter_scale=data.get('ip_adapter_scale', 0.8),
            controlnet_scale=data.get('controlnet_scale', 0.8),
        )
        encoded = []
        for img in images:
            buf = io.BytesIO()
            img.save(buf, format='PNG')
            encoded.append(base64.b64encode(buf.getvalue()).decode())
        return jsonify({'images': encoded, 'count': len(encoded)})
    except Exception as exc:
        return jsonify({'error': str(exc)}), 500


server_thread = threading.Thread(
    target=lambda: flask_app.run(host='0.0.0.0', port=5000, debug=False)
)
server_thread.daemon = True
server_thread.start()
print('Flask server started on :5000')

In [ ]:
# Cell 6 — Cloudflared tunnel — run last, keep this cell alive
# The URL printed below is what goes in .env as INSTANTID_SERVER_URL.
# Keepalive loop prevents Kaggle idle-timeout from killing the session.
import os, re, subprocess, time, sys

cloudflared = '/tmp/cloudflared'
if not os.path.exists(cloudflared):
    subprocess.run([
        'wget', '-q',
        'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
        '-O', cloudflared,
    ], check=True)
    os.chmod(cloudflared, 0o755)


def start_tunnel():
    p = subprocess.Popen(
        [cloudflared, 'tunnel', '--url', 'http://localhost:5000'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    url = None
    for line in p.stdout:
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if m:
            url = m.group(0)
            break
    return p, url


print('Waiting for tunnel...')
proc, tunnel_url = start_tunnel()

print()
print('=' * 60)
print(f'INSTANTID_SERVER_URL={tunnel_url}')
print('Add the above line to your .env file on the GTA server.')
print('=' * 60)
sys.stdout.flush()

# Keep session alive — restart cloudflared if it dies, print heartbeat every 60s
while True:
    if proc.poll() is not None:
        print(f'[{time.strftime("%H:%M:%S")}] cloudflared exited, restarting...')
        sys.stdout.flush()
        proc, new_url = start_tunnel()
        if new_url and new_url != tunnel_url:
            tunnel_url = new_url
            print(f'[{time.strftime("%H:%M:%S")}] New tunnel URL: {tunnel_url}')
            sys.stdout.flush()
    print(f'[{time.strftime("%H:%M:%S")}] alive — {tunnel_url}')
    sys.stdout.flush()
    time.sleep(60)